# Recap


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Recap en uitbreiding

In Q1 heb je geleerd om een botsend deeltjes model te maken, startend met 1 of 2 deeltjes en dit vervolgens uitbreidend naar ~100 deeltjes. Daarbij was het logisch om gebruik te maken van classes.

Logischerwijs zou onze simulatie de volgende stappen doorlopen:
1. aanmaken van deeltjes
2. bepalen van randvoorwaarden en initiele condities
3. simulaties bestaande uit:
    - bepaling van positie van deeltjes
    - controleren op onderlinge botsingen 
    - controleren op bostingen met wanden
    - aanpassen van eigenschappen zoals totaal volume of temperatuur
    - opslaan van gegevens
    
```{note}
Alhoewel het interessant is om steeds een animatie te maken van de beweging van de deeltjes, vraagt dit veel rekentijd. Je zou ook regelmatig een plot kunnen laten maken om te controleren of het programma werkt zoals je zou mogen verwachten.
```

### Deel 1: Particle class
Ons particle class definieert van een deeltje de massa $m$, snelheid $v$, positie $r$, en straal $R$. De positie moet per tijdstap bepaald worden. Let op! We geven alvast de code voor zwaartekracht, maar door de comment wordt deze niet gebruikt.

Daarnaast heeft de ParticleClass een functie die bekijkt of er een botsing plaats vindt.

Nieuw aan de ParticleClass zijn twee properties, nl. momentum en energy. Een property is eigenlijk een functie die zich gedraagt als eigenschap. Zo geeft particle.kin_energy de kinetische energie terug van het deeltje.

In [ ]:
# Maken van de class
class ParticleClass:
    # Het maken van het deeltje
    def __init__(self, m, v, r, R):
        self.m = m                         
        self.v = np.array(v, dtype=float)  
        self.r = np.array(r, dtype=float)  
        self.R = np.array(R, dtype=float)  

    # Het updaten van de positie, eventueel met zwaartekracht
    def update_position(self, dt):
        self.r += self.v * dt # + 1/2 * a * dt**2  
    
    # Het updaten van de snelheid door zwaartekracht
    # def update_velocity(self, a, dt):
    #     """Update the particle's velocity."""
    #     self.v += a*dt
    
    # Het bepalen of er een botsing plaats vindt
    def collide_detection(self, other):
        return np.linalg.norm(self.r - other.r) < (self.R + other.R)
    
    # Harde wand
    def boxcollision(self):
        if abs(self.r[0]) + self.R > Box_length: 
            self.v[0] = -self.v[0]                                  # Omdraaien van de snelheid
            self.r[0] = np.sign(self.r[0]) * (Box_length - self.R)  # Zet terug net binnen box                 
        if abs(self.r[1]) + self.R > Box_length: 
            self.v[1] = -self.v[1]     
            self.r[1] = np.sign(self.r[1]) * (Box_length - self.R) 
            
    @property
    def momentum(self):
        return self.m * self.v
    
    @property
    def kin_energy(self):
        return 1/2 * self.m * np.dot(self.v, self.v)

### Deel 2: Bepalen van randvoorwaarden en initiele condities

- Ons model heeft $N$ aantal deeltjes. 
- We maken gebruik van een 2D simulatie.
- De deeltjes bevinden zich in een harde, vierkante doos.
- De tijdstap is relatief klein (d.w.z. de afgelegde weg ($v_0*dt$ is klein in vergelijking met de boxsize.

```{exercise}
Leg uit dat d.m.v. onderstaande code de deeltjes allemaal dezelfde snelheid hebben en uniform verdeeld zijn over de box.
```

```{exercise}
Leg uit dat het aanmaken van de box met de deeltjes met hun initiele posities en snelheden een best practices is.
```


In [ ]:
# Aanmaken van de randvoorwaarden en initiele condities
Box_size_0 = 10
Box_length_0 = Box_size_0/2
Box_length = Box_length_0     # De grootte van de box kan wijzigen!

# Particles
particles = []
N = 500
v_0 = 1

dt = 0.04

# Aanmaken van deeltjes
for i in range(N):
    vx = np.random.uniform(-v_0,v_0)
    vy = np.random.choice([-1, 1])*np.sqrt(v_0**2-vx**2)        
    pos = Box_length_0*np.random.uniform(-1,1,2)
    particles.append(ParticleClass(m=1.0, v=[vx, vy], r = pos, R=.5))
    



In [ ]:
plt.figure()

plt.xlabel('x')
plt.ylabel('y')

plt.xlim(-Box_length_0,Box_length_0)
plt.ylim(-Box_length_0,Box_length_0)


for particle, particle_object in enumerate(particles):
    plt.plot(particle_object.r[0],particle_object.r[1],'k.')
    plt.arrow(particle_object.r[0],particle_object.r[1], 
              particle_object.v[0],particle_object.v[1], 
              head_width=0.05, head_length=0.1, color='red')
plt.show()


### Onderlinge botsingen
Voor elk deeltje zou bepaald moeten worden of het botst. Als een deeltje gebotst is op een ander deeltje, dan mag dat tweede deeltej op de ignor list, want alleen enkele botsingen


In [ ]:
def handle_collisions(particles):
    ignore_list = []
    for i, p1 in enumerate(particles):
        if p1 in ignore_list:
            continue
        for j, p2 in enumerate(particles):
            if p1 is p2:
                continue
            if p1.collide_detection(p2):
                r1, r2 = p1.r, p2.r
                v1, v2 = p1.v, p2.v
                m1, m2 = p1.m, p2.m

                delta_r = r1 - r2
                delta_v = v1 - v2
                distance_squared = np.dot(delta_r, delta_r) + 1e-12  # voorkom deling door 0

                # Botsing oplossen volgens elastische botsing in 2D
                v1_new = v1 - 2 * m2 / (m1 + m2) * np.dot(delta_v, delta_r) / distance_squared * delta_r
                v2_new = v2 - 2 * m1 / (m1 + m2) * np.dot(-delta_v, -delta_r) / distance_squared * (-delta_r)

                p1.v = v1_new
                p2.v = v2_new

                ignore_list.append(p2)

In [ ]:
# Deel 3: draaien van de simulatie

Hier een tekst over het draaien van de simulatie

In [ ]:
for i in range(100):
    
    for p in particles:
        p.update_position(dt)
        p.boxcollision()  # Wandbotsing werkt per deeltje
    handle_collisions(particles)


Zoals aangegeven, we kunnen een animatie maken van de positie en snelheid als functie van de tijd, maar we kunnen ook het eindresultaat tonen en interpreteren:

In [ ]:
plt.figure()

plt.xlabel('x')
plt.ylabel('y')

plt.xlim(-Box_length_0,Box_length_0)
plt.ylim(-Box_length_0,Box_length_0)


for particle, particle_object in enumerate(particles):
    plt.plot(particle_object.r[0],particle_object.r[1],'k.')
    plt.arrow(particle_object.r[0],particle_object.r[1], 
              particle_object.v[0],particle_object.v[1], 
              head_width=0.05, head_length=0.1, color='red')
plt.show()


In [ ]:
en dan toch hier de animatie